In [ ]:
import os,sys
import time
import numpy as np
import scipy.io as scio
import tensorflow as tf
import keras
import random
import gc
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, Conv2D, MaxPooling2D, Bidirectional, LSTM, GRU, Activation, TimeDistributed, Flatten, Softmax, BatchNormalization, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import KLDivergence, CategoricalCrossentropy
from sklearn.metrics import (accuracy_score, precision_score,cohen_kappa_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)
from sklearn.model_selection import train_test_split
import tensorflow_model_optimization as tfmot

In [ ]:
# Rebuild TEACHER with separated logits layer
def assemble_model(input_shape, n_class):
    model_input = Input(shape=input_shape, dtype='float32',
                        name='name_model_input')

    x = TimeDistributed(Conv2D(32, kernel_size=(5,5), activation='relu',
                               data_format='channels_last'))(model_input)
    x = TimeDistributed(MaxPooling2D(pool_size=(2,2)))(x)
    x = TimeDistributed(Flatten())(x)
    x = TimeDistributed(Dense(256, activation='gelu'))(x)
    x = Bidirectional(GRU(128, return_sequences=True))(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = Flatten()(x)

    # raw logits
    logits = Dense(n_class, name='logits')(x)                    
    output = Activation('softmax', name='name_model_output')(logits)
    

    model = Model(inputs=model_input, outputs=output)
    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=0.001),
        loss=loss_fn,
        metrics=['accuracy']
    )
    return model


In [ ]:
# Student model with Logits
# MultiHeadAttention Layer
# Create positional encoding
def positional_encoding(seq_len, d_model):
    angles = 1 / np.power(10000., (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    positional_encoding = np.zeros((seq_len, d_model))
    positional_encoding[:, 0::2] = np.sin(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 0::2]))
    positional_encoding[:, 1::2] = np.cos(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 1::2]))
    positional_encoding = positional_encoding[np.newaxis, ...]
    return tf.cast(positional_encoding, dtype=tf.float32)

# Create MultiHeadAttention Encoder
@tf.keras.utils.register_keras_serializable(package='Custom', name='ScaledDotProductAttention')
class ScaledDotProductAttention(Layer):
    def __init__(self, dropout_ratio=0.1, **kwargs):
        super(ScaledDotProductAttention,
              self).__init__(**kwargs)
        self.dropout_ratio = dropout_ratio  
        self.dropout       = Dropout(dropout_ratio)

    def call(self, q, k, v,
             mask=None, training=False):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = (
            matmul_qk / tf.math.sqrt(dk)
        )

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)
        output = tf.matmul(attention_weights, v)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            'dropout_ratio': self.dropout_ratio
        })
        return config


# MultiHeadAttention
@tf.keras.utils.register_keras_serializable(package='Custom', name='MultiHeadAttention')
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads,
                 dropout_ratio=0.1, **kwargs):
        super(MultiHeadAttention,
              self).__init__(**kwargs)
        self.num_heads     = num_heads
        self.d_model       = d_model
        self.dropout_ratio = dropout_ratio

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq    = Dense(d_model)
        self.wk    = Dense(d_model)
        self.wv    = Dense(d_model)

        # Create ONCE in __init__, not in call()
        self.attention = ScaledDotProductAttention(dropout_ratio)
        self.dropout   = Dropout(dropout_ratio)
        self.dense     = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(
            x,
            (batch_size, -1,
             self.num_heads, self.depth)
        )
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v,
             mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # Reuse existing instance
        scaled_attention = self.attention( q, k, v, mask, training)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention,(batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model'      : self.d_model,
            'num_heads'    : self.num_heads,
            'dropout_ratio': self.dropout_ratio
        })
        return config


# TransformerMultiHeadAttentionEncoderLayer
@tf.keras.utils.register_keras_serializable(package='Custom', name='TransformerMultiHeadAttentionEncoderLayer')
class TransformerMultiHeadAttentionEncoderLayer(Layer):
    def __init__(self, ff_dim,
                 dropout_ratio, **kwargs):
        super(
            TransformerMultiHeadAttentionEncoderLayer,
            self
        ).__init__(**kwargs)
        self.ff_dim        = ff_dim
        self.dropout_ratio = dropout_ratio

        # Move Dropout to __init__
        self.dropout1 = Dropout(dropout_ratio)
        self.dropout2 = Dropout(dropout_ratio)

    def build(self, input_shape):
        d_model = input_shape[-1]

        self.attention   = MultiHeadAttention(
            d_model       = d_model,
            num_heads     = 4,
            dropout_ratio = self.dropout_ratio
        )
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dense1      = Dense(
            self.ff_dim,
            activation         = 'gelu',
            kernel_regularizer = l2(0.01)
        )
        self.dense2      = Dense(
            d_model,
            kernel_regularizer = l2(0.01)
        )
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        super(
            TransformerMultiHeadAttentionEncoderLayer,
            self
        ).build(input_shape)

    def call(self, inputs, training=False):
        seq_len = inputs.shape[1]
        d_model = inputs.shape[2]

        pos_encoding = positional_encoding(seq_len, d_model)
        x = inputs + pos_encoding

        attn_output = self.attention(x, x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            'ff_dim'       : self.ff_dim,
            'dropout_ratio': self.dropout_ratio
        })
        return config

    
def transformer_model(input_shape, n_class, num_layers=2):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')
    
    # Feature extraction part
    x = TimeDistributed(Flatten())(model_input)
    x = TimeDistributed(Dense(128, activation='gelu'))(x)
    x = TimeDistributed(Dropout(0.5))(x)
    
    # Transformer encoder stack
    for _ in range(num_layers):
        x = TransformerMultiHeadAttentionEncoderLayer(ff_dim=256, dropout_ratio=0.1)(x)
    
    # Output part
    x = Flatten()(x)
    logits = Dense(n_class, name='logits')(x) 
    output = Activation('softmax', name='name_model_output')(logits)
    
    # Model compiling
    model = Model(inputs=model_input, outputs=output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.0005),
                  loss = loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_2 = transformer_model(input_shape=(36, 20, 20, 1), n_class=6, num_layers=2)

In [ ]:
# Load saved teacher and extract logit sub-model
def load_teacher_with_logits(filepath):

    teacher_model           = tf.keras.models.load_model(filepath)
    teacher_model.trainable = False

    # ── Inspect the final layer ───────────────────────────────
    final_layer = teacher_model.layers[-1]
    print(f"\n[Teacher] Final layer    : {final_layer.name}")
    print(f"[Teacher] Final layer cfg: {final_layer.get_config()}")
    layer_names = [l.name for l in teacher_model.layers]
    print(f"[Teacher] All layers: {layer_names}")

    if 'logits' in layer_names:
        print("[Teacher] Found 'logits' layer → direct extraction")
        teacher_logit_model = tf.keras.Model(
            inputs  = teacher_model.input,
            outputs = teacher_model.get_layer('logits').output
        )

    else:
        print("[Teacher] No 'logits' layer → rebuilding logit extractor")
        pre_final_output = teacher_model.layers[-2].output
        fused_weights = final_layer.get_weights()   
        n_class       = fused_weights[1].shape[0]   

        # New Dense WITHOUT activation — same weights
        logit_layer   = Dense(n_class, name='logits', use_bias=True)
        logit_output  = logit_layer(pre_final_output)

        # Build the logit extractor model
        teacher_logit_model = tf.keras.Model(
            inputs  = teacher_model.input,
            outputs = logit_output)

        # Copy weights from fused layer to new logit layer
        teacher_logit_model.get_layer('logits').set_weights(fused_weights)
        print(f"[Teacher] Weights copied from '{final_layer.name}' → 'logits'")

    teacher_logit_model.trainable = False

    print(f"[Teacher] prob  output : {teacher_model.output.shape}")
    print(f"[Teacher] logit output : {teacher_logit_model.output.shape}")
    print("\n[Sanity Check] Running on dummy input...")
    dummy        = tf.zeros((1,) + tuple(teacher_model.input.shape[1:]))
    prob_direct  = teacher_model(dummy,       training=False).numpy()
    logits_out   = teacher_logit_model(dummy, training=False).numpy()
    prob_from_lg = tf.nn.softmax(logits_out).numpy()
    max_diff     = float(np.max(np.abs(prob_direct - prob_from_lg)))
    print(f"  max |softmax(logits) - model_probs| = {max_diff:.2e}")
    if max_diff < 1e-5:
        print("  Logit extractor is correct")
    else:
        print(" Difference detected — check layer extraction")

    return teacher_model, teacher_logit_model

# Build student logits
def build_student_with_logits(input_shape, n_class):

    student_model = transformer_model(input_shape, n_class, num_layers=2)

    student_logit_model = tf.keras.Model(
        inputs  = student_model.input,
        outputs = student_model.get_layer('logits').output )

    print(f"[Student built]")
    print(f"  Full model output shape  : {student_model.output.shape}")
    print(f"  Logit model output shape : {student_logit_model.output.shape}")

    return student_model, student_logit_model


In [ ]:
# KD Functions
def distillation_loss(teacher_logits, student_logits, temperature):

    teacher_soft = tf.nn.softmax(teacher_logits / temperature)
    student_soft = tf.nn.softmax(student_logits / temperature)
    return KLDivergence()(teacher_soft, student_soft)


def hard_loss_fn(true_labels, student_probs):

    return CategoricalCrossentropy(
        label_smoothing=0.1,
        from_logits=False
    )(true_labels, student_probs)


def combined_loss(teacher_logits, student_logits,
                  student_probs,  true_labels,
                  temperature,    alpha):
    soft_loss = distillation_loss(teacher_logits, student_logits, temperature)
    hard_loss = hard_loss_fn(true_labels, student_probs)
    return alpha * hard_loss + (1 - alpha) * (temperature ** 2) * soft_loss


def compute_balanced_alpha(teacher_logit_model, student_model,
                           student_logit_model, training_generator,
                           temperature, n_batches=5):

    hard_vals = []
    soft_vals = []

    for i, (batch_data, batch_labels) in enumerate(training_generator):
        if i >= n_batches:
            break
        t_logits = teacher_logit_model(batch_data, training=False)
        s_logits = student_logit_model(batch_data, training=False)
        s_probs  = student_model(batch_data,       training=False)

        hard_vals.append(float(hard_loss_fn(batch_labels, s_probs)))
        soft_vals.append(float(distillation_loss(t_logits, s_logits, temperature)))

    mean_hard   = np.mean(hard_vals)
    mean_soft   = np.mean(soft_vals)
    t2_soft     = (temperature ** 2) * mean_soft
    alpha       = float(t2_soft / (mean_hard + t2_soft))

    print(f"\n[Alpha Estimation over {n_batches} batches]")
    print(f"  mean hard loss        : {mean_hard:.4f}")
    print(f"  mean soft loss        : {mean_soft:.4f}")
    print(f"  T²× mean soft loss    : {t2_soft:.4f}")
    print(f"  → balanced alpha      : {alpha:.4f}")
    return alpha

In [ ]:
# Train/Val
def train_step(batch_data, batch_labels,
               teacher_logit_model, student_model,
               student_logit_model, optimizer,
               temperature, alpha, acc_metric):

    with tf.GradientTape() as tape:
        teacher_logits = teacher_logit_model(batch_data, training=False)
        student_logits = student_logit_model(batch_data, training=True)
        student_probs  = student_model(batch_data,       training=True)
        loss = combined_loss(
            teacher_logits, student_logits,
            student_probs,  batch_labels,
            temperature,    alpha
        )

    grads = tape.gradient(loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))
    acc_metric.update_state(batch_labels, student_probs)
    return loss


def val_step(batch_data, batch_labels,
             teacher_logit_model, student_model,
             student_logit_model, temperature,
             alpha, acc_metric):

    teacher_logits = teacher_logit_model(batch_data, training=False)
    student_logits = student_logit_model(batch_data, training=False)
    student_probs  = student_model(batch_data,       training=False)
    loss = combined_loss(
        teacher_logits, student_logits,
        student_probs,  batch_labels,
        temperature,    alpha
    )
    acc_metric.update_state(batch_labels, student_probs)
    return loss

# Evaluate Functions
def evaluate_student(student_model, generator, split_name='test'):

    all_true  = []
    all_preds = []

    for batch_data, batch_labels in generator:
        probs      = student_model(batch_data, training=False).numpy()
        pred_class = np.argmax(probs,        axis=1)
        true_class = np.argmax(batch_labels, axis=1)
        all_preds.extend(pred_class)
        all_true.extend(true_class)

    all_true  = np.array(all_true)
    all_preds = np.array(all_preds)

    results = {
        'accuracy'            : float(accuracy_score(all_true, all_preds)),
        'precision_weighted'  : float(precision_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'recall_weighted'     : float(recall_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'f1_weighted'         : float(f1_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'precision_macro'     : float(precision_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'recall_macro'        : float(recall_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'f1_macro'            : float(f1_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'confusion_matrix'    : confusion_matrix(all_true, all_preds).tolist(),
        'classification_report': classification_report(
                                    all_true, all_preds,
                                    zero_division=0,
                                    output_dict=True)
    }

    # Print summary
    print(f"\n  [{split_name}]")
    for k, v in results.items():
        if k not in ('confusion_matrix', 'classification_report'):
            print(f"    {k:25s}: {v:.4f}")

    return results

In [ ]:
# Train loop function: FP32 Teacher-FP32 Student
SEEDS            = [42, 100, 200, 300, 500]
TEMPERATURE      = 15
EPOCHS           = 50
LR               = 0.0005
SEEDS            = [42, 100, 200, 300, 500]
all_seeds_results = {}

for seed in SEEDS:
    tf.random.set_seed(seed)
    np.random.seed(seed)

    # ── Teacher ───────────────────────────────────────────────
    teacher_path = os.path.join(SAVE_DIR, f'model_1_seed{seed}.keras')
    teacher_model, teacher_logit_model = load_teacher_with_logits(teacher_path)

    # ── Student ───────────────────────────────────────────────
    student_model, student_logit_model = build_student_with_logits(
        input_shape=(36, 20, 20, 1), n_class=6
    )
    optimizer = keras.optimizers.RMSprop(learning_rate=LR)

    # ── Auto alpha ────────────────────────────────────────────
    alpha = compute_balanced_alpha(
        teacher_logit_model, student_model,
        student_logit_model, training_generator,
        temperature=TEMPERATURE, n_batches=5
    )

    # ── Metrics ───────────────────────────────────────────────
    train_acc = tf.keras.metrics.CategoricalAccuracy()
    val_acc   = tf.keras.metrics.CategoricalAccuracy()
    history   = {'train_loss': [], 'train_acc': [],
                 'val_loss'  : [], 'val_acc'  : []}
    best_val_acc = 0.0
    student_path = os.path.join(KD_MODELS_DIR, f'student_seed{seed}.weights.h5')
    student_model.save_weights(student_path)            
    
    # To load: rebuild architecture first, then load weights
    best_student = transformer_model(
        input_shape=(36, 20, 20, 1), n_class=6, num_layers=2)
    
    best_student.load_weights(student_path) 

    # ── Epoch loop ────────────────────────────────────────────
    for epoch in range(EPOCHS):
        train_acc.reset_state()
        val_acc.reset_state()
        train_loss_sum = val_loss_sum = 0.0

        for step, (bd, bl) in enumerate(training_generator):
            loss = train_step(bd, bl, teacher_logit_model,
                              student_model, student_logit_model,
                              optimizer, TEMPERATURE, alpha, train_acc)
            train_loss_sum += loss.numpy()

        for v_step, (bd, bl) in enumerate(validation_generator):
            loss = val_step(bd, bl, teacher_logit_model,
                            student_model, student_logit_model,
                            TEMPERATURE, alpha, val_acc)
            val_loss_sum += loss.numpy()

        t_acc = float(train_acc.result())
        v_acc = float(val_acc.result())
        history['train_loss'].append(train_loss_sum / (step   + 1))
        history['train_acc'].append(t_acc)
        history['val_loss'].append(val_loss_sum  / (v_step + 1))
        history['val_acc'].append(v_acc)

        print(f"  Epoch {epoch+1:02d}/{EPOCHS} | "
              f"Train Loss: {train_loss_sum/(step+1):.4f} Acc: {t_acc:.4f} | "
              f"Val Loss: {val_loss_sum/(v_step+1):.4f} Acc: {v_acc:.4f}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            student_model.save(student_path)
            print(f"  [Saved] {student_path}")

    # ── Load best & evaluate ──────────────────────────────────
    best_student = tf.keras.models.load_model(student_path)

    seed_result = {
        'seed'        : seed,
        'alpha_used'  : float(alpha),
        'best_val_acc': float(best_val_acc),
        'history'     : history,
        'test'        : evaluate_student(best_student, test_generator,   'test'),
        'test_1'      : evaluate_student(best_student, test_1_generator, 'test_1'),
        'test_2'      : evaluate_student(best_student, test_2_generator, 'test_2'),
    }
    all_seeds_results[f'seed_{seed}'] = seed_result

for split in ['test', 'test_1', 'test_2']:
    print(f"\n  [{split}]")
    for metric in ['accuracy', 'f1_weighted',
                   'precision_weighted', 'recall_weighted']:
        values = [all_seeds_results[f'seed_{s}'][split][metric]
                  for s in SEEDS]
        print(f"    {metric:25s}: mean={np.mean(values):.4f} "
              f"std={np.std(values):.4f} "
              f"per_seed={[round(v,4) for v in values]}")

TF.Lite Student

In [ ]:

test_sets_dict = {
    'all_rooms': (data_test,   label_test),
    'room2'    : (data_test_1, label_test_1),
    'room3'    : (data_test_2, label_test_2), }

# CUSTOM OBJECTS for load_model()
CUSTOM_OBJECTS = {
    'ScaledDotProductAttention'               : ScaledDotProductAttention,
    'MultiHeadAttention'                      : MultiHeadAttention,
    'TransformerMultiHeadAttentionEncoderLayer': TransformerMultiHeadAttentionEncoderLayer
}

# Representive Dataset
N_CALIB_SAMPLES = 300
def get_calib_data(train_data, seed, n_samples):
    rng   = np.random.default_rng(seed)
    idx   = rng.choice(len(train_data),
                       size    = min(n_samples, len(train_data)),
                       replace = False)
    calib = train_data[idx].astype(np.float32)
    print(f"  [Calib] {len(calib)} samples drawn (seed={seed})")
    return calib


def make_representative_dataset(calib_data, seed=42):
    calib_subset = calib_data.astype(np.float32)
    print(f"    [Calib] {len(calib_subset)} samples ready (seed={seed})")

    def representative_dataset():
        for i in range(len(calib_subset)):
            yield [calib_subset[i:i+1]]      

    return representative_dataset


# Export FP32 Student to PTQ INT8 Student
def export_tflite_full_int8(keras_model,
                             save_path,
                             representative_dataset_fn,
                             keep_float_io=True):

    print(f"    [TFLite Full-INT8] Calibrating + Quantizing...")

    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)

    # Quantization settings 
    converter.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset    = representative_dataset_fn
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.SELECT_TF_OPS     
    ]

    # Keep float32 I/O for easy inference 
    if keep_float_io:
        converter.inference_input_type  = tf.float32
        converter.inference_output_type = tf.float32

    converter._experimental_lower_tensor_list_ops = False

    try:
        tflite_model = converter.convert()
        with open(save_path, 'wb') as f:
            f.write(tflite_model)

        size_mb = os.path.getsize(save_path) / (1024 ** 2)
        print(f"    [✓] Saved : {save_path}")
        print(f"    [✓] Size  : {size_mb:.3f} MB")
        return size_mb

    except Exception as e:
        print(f"    [ERROR] Full-INT8 export failed: {e}")
        return -1


# Evalaute PTQ Student
def evaluate_tflite(tflite_path, test_data, test_labels,
                    label_name, n_warmup=5):

    if not os.path.exists(tflite_path):
        print(f"      [SKIP] Not found: {os.path.basename(tflite_path)}")
        return None

    interpreter = tf.lite.Interpreter(model_path  = tflite_path,
                                      num_threads = 1)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype    = input_details[0]['dtype']
    output_dtype   = output_details[0]['dtype']
    in_scale,  in_zp  = input_details[0]['quantization']
    out_scale, out_zp = output_details[0]['quantization']

    def preprocess(sample_f32):
        if input_dtype == np.int8 and in_scale != 0:
            return (sample_f32 / in_scale + in_zp).astype(np.int8)
        return sample_f32

    def postprocess(output):
        if output_dtype == np.int8 and out_scale != 0:
            return (output.astype(np.float32) - out_zp) * out_scale
        return output.astype(np.float32)

    for i in range(min(n_warmup, len(test_data))):
        sample = preprocess(test_data[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()

    preds   = []
    t_start = time.perf_counter()

    for i in range(len(test_data)):
        sample = preprocess(test_data[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(
            output_details[0]['index']).copy()
        preds.append(postprocess(output))

    t_end   = time.perf_counter()
    t_total = t_end - t_start
    n       = len(test_data)
    lat_ms  = (t_total / n) * 1000
    tput    = n / t_total
    y_prob = np.concatenate(preds, axis=0)
    y_true = np.argmax(test_labels, axis=1)
    y_pred = np.argmax(y_prob,      axis=1)

    present_classes = np.union1d(np.unique(y_true), np.unique(y_pred))
    n_classes       = len(present_classes)
    target_names    = [f'Gesture_{c+1}' for c in present_classes]

    acc    = float(accuracy_score(y_true, y_pred))
    prec_w = float(precision_score(y_true, y_pred, average='weighted',
                                   zero_division=0, labels=present_classes))
    rec_w  = float(recall_score(y_true, y_pred, average='weighted',
                                zero_division=0, labels=present_classes))
    f1_w   = float(f1_score(y_true, y_pred, average='weighted',
                             zero_division=0, labels=present_classes))
    prec_m = float(precision_score(y_true, y_pred, average='macro',
                                   zero_division=0, labels=present_classes))
    rec_m  = float(recall_score(y_true, y_pred, average='macro',
                                zero_division=0, labels=present_classes))
    f1_m   = float(f1_score(y_true, y_pred, average='macro',
                             zero_division=0, labels=present_classes))
    kappa  = float(cohen_kappa_score(y_true, y_pred))

    size_mb = os.path.getsize(tflite_path) / (1024 ** 2)

    print(f"      [{label_name}] "
          f"Acc={acc:.4f}  F1w={f1_w:.4f}  F1m={f1_m:.4f}  "
          f"κ={kappa:.4f}  Lat={lat_ms:.2f}ms  Size={size_mb:.3f}MB")

    return {
        "accuracy"                        : round(acc,    6),
        "precision_weighted"              : round(prec_w, 6),
        "recall_weighted"                 : round(rec_w,  6),
        "f1_weighted"                     : round(f1_w,   6),
        "precision_macro"                 : round(prec_m, 6),
        "recall_macro"                    : round(rec_m,  6),
        "f1_macro"                        : round(f1_m,   6),
        "kappa"                           : round(kappa,  6),
        "n_classes_detected"              : int(n_classes),
        "confusion_matrix"                : confusion_matrix(
                                                y_true, y_pred,
                                                labels=present_classes
                                            ).tolist(),
        "classification_report"           : classification_report(
                                                y_true, y_pred,
                                                labels=present_classes,
                                                target_names=target_names,
                                                digits=4,
                                                zero_division=0
                                            ),
        "total_inference_time_s"          : round(t_total, 4),
        "avg_inference_time_per_sample_ms": round(lat_ms,  4),
        "throughput_samples_per_sec"      : round(tput,    2),
        "model_size_mb"                   : round(size_mb, 4)
    }



# PTQ function for per model in per seed
def tflite_ptq_one_model(keras_model, model_tag,
                          calibration_data, test_sets_dict,
                          calib_seed, fp32_model_path=None):

    print(f"\n  {'═'*52}")
    print(f"  TFLite PTQ: {model_tag}")
    print(f"  {'═'*52}")

    tflite_path  = os.path.join(TFLITE_DIR,
                                f'{model_tag}_full_int8.tflite')

    fp32_size_mb = None
    if fp32_model_path and os.path.exists(fp32_model_path):
        fp32_size_mb = os.path.getsize(fp32_model_path) / (1024 ** 2)
        print(f"\n  [FP32 Reference] Size: {fp32_size_mb:.3f} MB")

    print(f"\n  [Step 1] Preparing calibration data...")
    rep_dataset_fn = make_representative_dataset(
        calib_data = calibration_data,
        seed       = calib_seed
    )
    print(f"\n  [Step 2] Exporting TFLite INT8...")
    int8_size_mb = export_tflite_full_int8(
        keras_model               = keras_model,
        save_path                 = tflite_path,
        representative_dataset_fn = rep_dataset_fn,
        keep_float_io             = True
    )

    print(f"\n  [Step 3] Size Summary:")
    if fp32_size_mb and int8_size_mb > 0:
        saved_mb       = fp32_size_mb - int8_size_mb
        reduction_pct  = saved_mb / fp32_size_mb * 100
        compress_ratio = fp32_size_mb / int8_size_mb
        print(f"    FP32  : {fp32_size_mb:.3f} MB")
        print(f"    INT8  : {int8_size_mb:.3f} MB")
        print(f"    Saved : {saved_mb:.3f} MB  "
              f"({reduction_pct:.1f}% smaller, "
              f"{compress_ratio:.2f}x compression)")
    else:
        print(f"    INT8  : {int8_size_mb:.3f} MB")

    print(f"\n  [Step 4] Evaluating on all test sets...")
    metrics = {}

    if int8_size_mb > 0:
        for ts_name, (ts_data, ts_labels) in test_sets_dict.items():
            metrics[ts_name] = evaluate_tflite(
                tflite_path = tflite_path,
                test_data   = ts_data,
                test_labels = ts_labels,
                label_name  = f"INT8|{ts_name}",
                n_warmup    = 5
            )
    else:
        print(" [SKIP] Export failed — no evaluation")

    result = {
        'model_tag'   : model_tag,
        'fp32_size_mb': fp32_size_mb,
        'int8_size_mb': int8_size_mb,
        'metrics'     : metrics
    }

    if fp32_size_mb and int8_size_mb > 0:
        result['size_reduction'] = {
            'fp32_mb'          : round(fp32_size_mb,  4),
            'int8_mb'          : round(int8_size_mb,  4),
            'saved_mb'         : round(saved_mb,       4),
            'reduction_pct'    : round(reduction_pct,  2),
            'compression_ratio': round(compress_ratio, 3)
        }

    return result


In [ ]:
# Evaluate PTQ Student per seed
all_tflite_kd_student = {}

for seed_idx, seed in enumerate(SEEDS):
    print(f"  KD STUDENT | SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    m_weights_path = os.path.join(
        KD_MODELS_DIR,
        f'student_seed{seed}.weights.h5'   
    )

    if not os.path.exists(m_weights_path):
        print(f"  [SKIP] Not found: {m_weights_path}")
        continue

    # Rebuild architecture + load weights
    student_model = transformer_model(
        input_shape = (36, 20, 20, 1),
        n_class     = 6,
        num_layers  = 2
    )
    student_model.load_weights(m_weights_path)
    student_model.trainable = False        
    print(f"  [✓] student_seed{seed} weights loaded")
    print(f"  [✓] Params: {student_model.count_params():,}")

    # ── Per-seed calibration data ─────────────────────────────
    calib_data = get_calib_data(
        train_data = data_train,
        seed       = seed,
        n_samples  = N_CALIB_SAMPLES
    )

    # ── fp32_model_path = None (weights.h5 ≠ FP32 model) ───
    all_tflite_kd_student[seed] = tflite_ptq_one_model(
        keras_model      = student_model,
        model_tag        = f'kd_student_seed{seed}',
        calibration_data = calib_data,
        test_sets_dict   = test_sets_dict,
        calib_seed       = seed,
        fp32_model_path  = None            
    )

for split in ['test', 'test_1', 'test_2']:
    print(f"\n  [{split}]")
    for metric in ['accuracy', 'f1_weighted',
                   'precision_weighted', 'recall_weighted']:
        values = []
        for s in SEEDS:
            try:
                v = all_tflite_kd_student[s]['metrics'][split][metric]
                values.append(v)
            except (KeyError, TypeError):
                pass
        if values:
            print(f"    {metric:25s}: "
                  f"mean={np.mean(values):.4f}  "
                  f"std={np.std(values):.4f}  "
                  f"per_seed={[round(v,4) for v in values]}")

TF.Lite QAT

In [ ]:
# Fake Quantization Layers
LastValueQuantizer     = tfmot.quantization.keras.quantizers.LastValueQuantizer
MovingAverageQuantizer = tfmot.quantization.keras.quantizers.MovingAverageQuantizer
quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
quantize_scope          = tfmot.quantization.keras.quantize_scope

class TimeDistributedQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):

    def get_weights_and_quantizers(self, layer):
        return []

    def get_activations_and_quantizers(self, layer):
        inner = layer.layer
        if (hasattr(inner, 'activation') and
                inner.activation is not None and
                inner.activation != tf.keras.activations.linear):
            return [(inner.activation,
                     MovingAverageQuantizer(num_bits=8, symmetric=False,
                                           narrow_range=False, per_axis=False))]
        return []

    def set_quantize_weights(self, layer, quantize_weights):
        pass  

    def set_quantize_activations(self, layer, quantize_activations):
        if quantize_activations:
            layer.layer.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}


class DenseQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):

    def get_weights_and_quantizers(self, layer):
        return [(layer.kernel,
                 LastValueQuantizer(num_bits=8, symmetric=True,
                                    narrow_range=False, per_axis=False))]

    def get_activations_and_quantizers(self, layer):
        if (hasattr(layer, 'activation') and
                layer.activation != tf.keras.activations.linear):
            return [(layer.activation,
                     MovingAverageQuantizer(num_bits=8, symmetric=False,
                                           narrow_range=False, per_axis=False))]
        return []

    def set_quantize_weights(self, layer, quantize_weights):
        layer.kernel = quantize_weights[0]

    def set_quantize_activations(self, layer, quantize_activations):
        if quantize_activations:
            layer.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}

class DenseLinearQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):

    def get_weights_and_quantizers(self, layer):
        return [(layer.kernel,
                 LastValueQuantizer(num_bits=8, symmetric=True,
                                    narrow_range=False, per_axis=False))]

    def get_activations_and_quantizers(self, layer):
        return []  # Linear → nothing to quantize

    def set_quantize_weights(self, layer, quantize_weights):
        layer.kernel = quantize_weights[0]

    def set_quantize_activations(self, layer, quantize_activations):
        pass

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}



class ActivationQuantizeConfig(tfmot.quantization.keras.QuantizeConfig):

    def get_weights_and_quantizers(self, layer):
        return []

    def get_activations_and_quantizers(self, layer):
        return []

    def set_quantize_weights(self, layer, quantize_weights):
        pass

    def set_quantize_activations(self, layer, quantize_activations):
        pass

    def get_output_quantizers(self, layer):
        return []  

    def get_config(self):
        return {}


class TransformerMultiHeadAttentionEncoderQuantizeConfig(
        tfmot.quantization.keras.QuantizeConfig):

    def get_weights_and_quantizers(self, layer):
        return [
            (layer.dense1.kernel,
             LastValueQuantizer(num_bits=8, symmetric=True,
                                narrow_range=False, per_axis=False)),
            (layer.dense2.kernel,
             LastValueQuantizer(num_bits=8, symmetric=True,
                                narrow_range=False, per_axis=False)),
        ]

    def get_activations_and_quantizers(self, layer):
        return [
            (layer.dense1.activation,
             MovingAverageQuantizer(num_bits=8, symmetric=False,
                                    narrow_range=False, per_axis=False)),
        ]

    def set_quantize_weights(self, layer, quantize_weights):
        layer.dense1.kernel = quantize_weights[0]
        layer.dense2.kernel = quantize_weights[1]

    def set_quantize_activations(self, layer, quantize_activations):
        layer.dense1.activation = quantize_activations[0]

    def get_output_quantizers(self, layer):
        return []

    def get_config(self):
        return {}

# Apply Fake Quantization
def apply_quantization(layer):

    if isinstance(layer, tf.keras.layers.TimeDistributed):
        return quantize_annotate_layer(layer, TimeDistributedQuantizeConfig())
    elif isinstance(layer, TransformerMultiHeadAttentionEncoderLayer):
        return quantize_annotate_layer(
            layer, TransformerMultiHeadAttentionEncoderQuantizeConfig())
    elif isinstance(layer, tf.keras.layers.Dense):
        if layer.name == 'logits':
            return quantize_annotate_layer(layer, DenseLinearQuantizeConfig())
        else:
            return quantize_annotate_layer(layer, DenseQuantizeConfig())
    elif isinstance(layer, tf.keras.layers.Activation):
        return quantize_annotate_layer(layer, ActivationQuantizeConfig())
    elif isinstance(layer, (tf.keras.layers.Flatten,
                             tf.keras.layers.Dropout)):
        return layer
    
    return layer


#  Build QAT student
def build_qat_student_with_logits(input_shape, n_class, lr=0.0005):

    # Build FP32 student first
    base_student = transformer_model(
        input_shape = input_shape,
        n_class     = n_class,
        num_layers  = 2
    )
    print(f"[QAT Build] Base FP32 params: {base_student.count_params():,}")

    # Annotate + apply QAT
    with quantize_scope({
        'ScaledDotProductAttention'                         : ScaledDotProductAttention,
        'MultiHeadAttention'                                : MultiHeadAttention,
        'TransformerMultiHeadAttentionEncoderLayer'         : TransformerMultiHeadAttentionEncoderLayer,
        'TimeDistributedQuantizeConfig'                     : TimeDistributedQuantizeConfig,
        'TransformerMultiHeadAttentionEncoderQuantizeConfig': TransformerMultiHeadAttentionEncoderQuantizeConfig,
        'DenseQuantizeConfig'                               : DenseQuantizeConfig,
        'DenseLinearQuantizeConfig'                         : DenseLinearQuantizeConfig,
        'ActivationQuantizeConfig'                          : ActivationQuantizeConfig,
    }):
        annotated = tf.keras.models.clone_model(
            base_student,
            clone_function=apply_quantization
        )
        qat_model = tfmot.quantization.keras.quantize_apply(annotated)

    # Find logits layer 
    logits_layer_name = None
    for layer in qat_model.layers:
        if 'logits' in layer.name:
            logits_layer_name = layer.name
            break

    if logits_layer_name is None:
        raise ValueError("Could not find 'logits' layer in QAT model!")

    print(f"[QAT Build] Found logits layer: {logits_layer_name}")

    qat_logits_output = qat_model.get_layer(logits_layer_name).output
    qat_probs_output  = qat_model.output

    qat_dual_model = tf.keras.Model(
        inputs  = qat_model.input,
        outputs = {
            'probs' : qat_probs_output,
            'logits': qat_logits_output
        }
    )

    qat_dual_model.compile(
        optimizer = keras.optimizers.RMSprop(learning_rate=lr),
        loss      = CategoricalCrossentropy(label_smoothing=0.1),
        metrics   = ['accuracy']
    )

    dummy   = tf.zeros((2,) + tuple(input_shape))
    outputs = qat_dual_model(dummy, training=False)
    probs_out  = outputs['probs'].numpy()
    logits_out = outputs['logits'].numpy()
    prob_sum   = probs_out.sum(axis=1).mean()

    print(f"[QAT Build] Prob sum   : {prob_sum:.4f}  (should be ~1.0)")

    n_zeros = int((probs_out < 1e-6).sum())
    print(f"[QAT Build] Zero probs : {n_zeros} / {probs_out.size}"
          f"  (should be 0 or very few)")

    if n_zeros > probs_out.size * 0.3:
        print(f"  ⚠ WARNING: {n_zeros} zero probabilities detected!")
        print(f"    → ActivationQuantizeConfig.get_output_quantizers()"
              f" must return []")

    recon_prob = tf.nn.softmax(logits_out).numpy()
    max_diff   = float(np.max(np.abs(probs_out - recon_prob)))
    print(f"[QAT Build] Round-trip : {max_diff:.2e}"
          f"  (softmax(logits) vs probs, should be < 0.01)")
    dummy_labels = np.ones((2, n_class), dtype=np.float32) / n_class
    dummy_hard   = float(hard_loss_fn_qat(dummy_labels, probs_out))
    print(f"[QAT Build] Dummy hard loss: {dummy_hard:.4f}"
          f"  (should be < 3.0, if > 5 → zero prob problem remains)")

    if dummy_hard > 5.0:
        print(f"  ⚠ hard_loss={dummy_hard:.2f} is too high!")
        print(f"    → softmax output is still being quantized to zeros")
        print(f"    → check ActivationQuantizeConfig")

    return qat_dual_model

In [ ]:

SEEDS            = [42, 100, 200, 300, 500]
TEMPERATURE_QAT  = 15    
WARMUP_EPOCHS    = 5     # FP32 pre-training before QAT to get weights correctly
QAT_EPOCHS       = 45   
LR               = 0.001
LR_QAT           = 0.0005 

# Train/Val for QAT STudent
def make_train_step(teacher_logit_model, fp32_student,
                    optimizer, acc_metric, loss_metric):
    @tf.function
    def train_step(batch_data, batch_labels, temperature, alpha):
        with tf.GradientTape() as tape:
            teacher_logits = teacher_logit_model(batch_data, training=False)
            student_probs  = fp32_student(batch_data, training=True)
            student_logits = tf.math.log(
                tf.clip_by_value(student_probs, 1e-7, 1.0)
            )
            loss = combined_loss(
                teacher_logits = teacher_logits,
                student_logits = student_logits,
                student_probs  = student_probs,
                true_labels    = batch_labels,
                temperature    = temperature,
                alpha          = alpha
            )
        grads = tape.gradient(loss, fp32_student.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(
            zip(grads, fp32_student.trainable_variables)
        )
        acc_metric.update_state(batch_labels, student_probs)
        loss_metric.update_state(loss)

    return train_step


def make_qat_train_step(teacher_logit_model, qat_dual_model,
                         optimizer, acc_metric, loss_metric):
    @tf.function
    def train_step(batch_data, batch_labels, temperature, alpha):
        with tf.GradientTape() as tape:
            teacher_logits = teacher_logit_model(batch_data, training=False)
            outputs        = qat_dual_model(batch_data, training=True)
            student_probs  = outputs['probs']
            student_logits = outputs['logits']

            loss = combined_loss_qat(      
                teacher_logits = teacher_logits,
                student_logits = student_logits,
                student_probs  = student_probs,
                true_labels    = batch_labels,
                temperature    = temperature,
                alpha          = alpha
            )

        grads = tape.gradient(loss, qat_dual_model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, clip_norm=1.0)
        optimizer.apply_gradients(
            zip(grads, qat_dual_model.trainable_variables)
        )
        acc_metric.update_state(batch_labels, student_probs)
        loss_metric.update_state(loss)

    return train_step


def make_qat_val_step(teacher_logit_model, qat_dual_model,
                       acc_metric, loss_metric):
    @tf.function
    def val_step(batch_data, batch_labels, temperature, alpha):
        teacher_logits = teacher_logit_model(batch_data, training=False)
        outputs        = qat_dual_model(batch_data, training=False)
        student_probs  = outputs['probs']
        student_logits = outputs['logits']

        loss = combined_loss_qat(           
            teacher_logits = teacher_logits,
            student_logits = student_logits,
            student_probs  = student_probs,
            true_labels    = batch_labels,
            temperature    = temperature,
            alpha          = alpha
        )
        acc_metric.update_state(batch_labels, student_probs)
        loss_metric.update_state(loss)

    return val_step



def warmup_fp32_student(teacher_logit_model,
                         training_generator,
                         input_shape, n_class,
                         warmup_epochs, lr,
                         temperature, seed):

    print(f"\n  [Warmup] FP32 student for {warmup_epochs} epochs...")

    fp32_student = transformer_model(
        input_shape=input_shape,
        n_class=n_class,
        num_layers=2
    )

    opt         = keras.optimizers.RMSprop(learning_rate=lr)
    acc_metric  = tf.keras.metrics.CategoricalAccuracy()
    loss_metric = tf.keras.metrics.Mean()

    temperature_tf = tf.constant(temperature, dtype=tf.float32)
    alpha_tf       = tf.constant(0.5,         dtype=tf.float32)

    # ── Compile ONCE per seed — captures this seed's models ───
    warmup_step = make_train_step(
        teacher_logit_model = teacher_logit_model,
        fp32_student        = fp32_student,
        optimizer           = opt,
        acc_metric          = acc_metric,
        loss_metric         = loss_metric
    )

    for epoch in range(warmup_epochs):
        acc_metric.reset_state()
        loss_metric.reset_state()

        for bd, bl in training_generator:
            warmup_step(bd, bl, temperature_tf, alpha_tf)

        print(f"    Warmup {epoch+1:02d}/{warmup_epochs} | "
              f"Loss: {float(loss_metric.result()):.4f} | "
              f"Acc:  {float(acc_metric.result()):.4f}")

    return fp32_student


def save_weights_npz(model, save_path):

    if os.path.exists(save_path):
        os.remove(save_path)

    weights_dict = {}
    for i, layer in enumerate(model.layers):
        for j, w in enumerate(layer.get_weights()):
            weights_dict[f'w_{i}_{j}'] = w

    np.savez(save_path, **weights_dict)
    size_kb = os.path.getsize(save_path) / 1024
    print(f"  [✓] Weights saved : {save_path} ({size_kb:.1f} KB)")


def load_weights_npz(model, load_path):
    data = np.load(load_path, allow_pickle=True)

    for i, layer in enumerate(model.layers):
        existing = layer.get_weights()
        if not existing:
            continue
        new_weights = []
        for j in range(len(existing)):
            key = f'w_{i}_{j}'
            if key in data:
                new_weights.append(data[key])
            else:
                new_weights.append(existing[j])   
        layer.set_weights(new_weights)

    print(f"  [✓] Weights loaded: {load_path}")
    return model



def transfer_fp32_weights_to_qat(fp32_model, qat_model):
    fp32_real_weights = []
    for layer in fp32_model.layers:
        for w in layer.get_weights():
            if len(w.shape) > 0:
                fp32_real_weights.append((layer.name, w))

    print(f"\n  [Transfer] FP32 real weight tensors: "
          f"{len(fp32_real_weights)}")
    for name, w in fp32_real_weights:
        print(f"    {name:50s} shape={w.shape}")

    print(f"\n  [Transfer] QAT layer weight slots:")
    for layer in qat_model.layers:
        ws = layer.get_weights()
        if ws:
            shapes = [w.shape for w in ws]
            print(f"    {layer.name:50s} slots={shapes}")

    fp32_idx    = 0
    transferred = 0

    for layer in qat_model.layers:
        if fp32_idx >= len(fp32_real_weights):
            break

        current_weights = layer.get_weights()
        if not current_weights:
            continue

        new_weights   = list(current_weights)
        layer_changed = False

        for slot_idx, slot_w in enumerate(current_weights):
            if fp32_idx >= len(fp32_real_weights):
                break
            if len(slot_w.shape) == 0:       
                continue

            fp32_name, fp32_w = fp32_real_weights[fp32_idx]

            if fp32_w.shape == slot_w.shape:
                new_weights[slot_idx] = fp32_w
                print(f"    [OK] fp32:{fp32_name} {fp32_w.shape}"
                      f" → qat:{layer.name}[{slot_idx}]")
                fp32_idx    += 1
                transferred += 1
                layer_changed = True
            else:
                print(f"    [MISMATCH] fp32:{fp32_name} {fp32_w.shape}"
                      f" ≠ qat:{layer.name}[{slot_idx}] {slot_w.shape}")
                fp32_idx += 1

        if layer_changed:
            layer.set_weights(new_weights)


    print(f"\n  [Transfer] Transferred : {transferred} / "
          f"{len(fp32_real_weights)} weight tensors")

    if transferred == len(fp32_real_weights):
        print(f"   All weights transferred successfully!")
    elif transferred > len(fp32_real_weights) * 0.7:
        print(f"  Most weights transferred ({transferred}/"
              f"{len(fp32_real_weights)})")
    else:
        print(f"   WARNING: Only {transferred}/"
              f"{len(fp32_real_weights)} transferred!")

    # Post-transfer sanity check 
    dummy    = tf.zeros((2,) + tuple(fp32_model.input_shape[1:]))
    fp32_out = fp32_model(dummy, training=False).numpy()
    qat_raw  = qat_model(dummy, training=False)
    if isinstance(qat_raw, dict):
        qat_out = qat_raw['probs'].numpy()
    else:
        qat_out = qat_raw.numpy()

    max_diff = float(np.max(np.abs(fp32_out - qat_out)))

    print(f"\n  [Transfer] FP32 probs : {fp32_out[0].round(4)}")
    print(f"  [Transfer] QAT  probs : {qat_out[0].round(4)}")
    print(f"  [Transfer] Max diff   : {max_diff:.4f}")

    if max_diff < 0.05:
        print(f" QAT output matches FP32 output after transfer")
    else:
        print(f" Large difference — quantization noise, "
              f"check if > 0.1")

    return qat_model


# Alpha estimation
def hard_loss_fn_qat(true_labels, student_probs):
    epsilon   = tf.constant(1e-4,  dtype=tf.float32)  
    smoothing = tf.constant(0.1,   dtype=tf.float32)  
    one       = tf.constant(1.0,   dtype=tf.float32)  
    probs_safe = tf.clip_by_value(student_probs, epsilon, one)
    probs_safe = probs_safe / tf.reduce_sum(
        probs_safe, axis=-1, keepdims=True)
    n_classes = tf.cast(tf.shape(true_labels)[-1], dtype=tf.float32)
    true_labels_f32 = tf.cast(true_labels, dtype=tf.float32)  
    smooth_labels = (one - smoothing) * true_labels_f32 + smoothing / n_classes
    log_probs = tf.math.log(probs_safe)
    ce        = -tf.reduce_sum(smooth_labels * log_probs, axis=-1)

    return tf.reduce_mean(ce)


def combined_loss_qat(teacher_logits, student_logits,
                      student_probs,  true_labels,
                      temperature,    alpha):

    alpha_f32 = tf.cast(alpha,       dtype=tf.float32)
    temp_f32  = tf.cast(temperature, dtype=tf.float32)

    soft_loss = distillation_loss(teacher_logits, student_logits, temp_f32)
    hard_loss = hard_loss_fn_qat(true_labels, student_probs)

    return (alpha_f32 * hard_loss
            + (tf.constant(1.0, dtype=tf.float32) - alpha_f32)
            * (temp_f32 ** 2)
            * soft_loss)



def compute_balanced_alpha_qat(teacher_logit_model,
                                qat_dual_model,
                                training_generator,
                                temperature,
                                n_batches=5):
    
    hard_vals = []
    soft_vals = []

    for i, (bd, bl) in enumerate(training_generator):
        if i >= n_batches:
            break

        t_logits = teacher_logit_model(bd, training=False)
        outputs  = qat_dual_model(bd, training=False)
        s_probs  = outputs['probs']
        s_logits = outputs['logits']
        h = float(hard_loss_fn_qat(bl, s_probs))
        s = float(distillation_loss(t_logits, s_logits, temperature))

        hard_vals.append(h)
        soft_vals.append(s)

    mean_hard = np.mean(hard_vals)
    mean_soft = np.mean(soft_vals)
    t2_soft   = (temperature ** 2) * mean_soft
    alpha     = float(t2_soft / (mean_hard + t2_soft + 1e-8))

    print(f"\n  [Alpha QAT] over {n_batches} batches")
    print(f"    mean hard loss     : {mean_hard:.4f}")
    print(f"    mean soft loss     : {mean_soft:.4f}")
    print(f"    T²× soft loss      : {t2_soft:.4f}")
    print(f"    → balanced alpha   : {alpha:.4f}")

    if mean_hard > 3.0:
        print(f"    hard_loss={mean_hard:.2f} still high after clipping")
        print(f"      → check if ActivationQuantizeConfig returns []")
        print(f"        for get_output_quantizers")

    if mean_soft < 0.001:
        print(f"    soft_loss={mean_soft:.5f} near zero")
        print(f"      → student logits may be near-zero (untrained)")
        print(f"      → KD signal very weak at this stage")

    if alpha < 0.3:
        print(f"   alpha={alpha:.4f} < 0.3 → using fallback 0.4")
        alpha = 0.4

    elif alpha > 0.9:
        print(f"    alpha={alpha:.4f} > 0.9 → using fallback 0.7")
        alpha = 0.7

    print(f"  → final alpha used : {alpha:.4f}")
    return alpha


# Evaluate functions for QAT Student
def evaluate_qat_student(student_model, generator, split_name='test'):
    all_true  = []
    all_preds = []

    for batch_data, batch_labels in generator:
        output = student_model(batch_data, training=False)
        if isinstance(output, dict):
            probs = output['probs'].numpy()
        else:
            probs = output.numpy()

        pred_class = np.argmax(probs,        axis=1)
        true_class = np.argmax(batch_labels, axis=1)
        all_preds.extend(pred_class)
        all_true.extend(true_class)

    results = {
        'accuracy'            : float(accuracy_score(all_true, all_preds)),
        'precision_weighted'  : float(precision_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'recall_weighted'     : float(recall_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'f1_weighted'         : float(f1_score(all_true, all_preds,
                                    average='weighted', zero_division=0)),
        'precision_macro'     : float(precision_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'recall_macro'        : float(recall_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'f1_macro'            : float(f1_score(all_true, all_preds,
                                    average='macro',    zero_division=0)),
        'confusion_matrix'    : confusion_matrix(all_true, all_preds).tolist(),
        'classification_report': classification_report(
                                    all_true, all_preds,
                                    zero_division=0,
                                    output_dict=True)}

    print(f"\n  [{split_name}]")
    for k, v in results.items():
        if k not in ('confusion_matrix', 'classification_report'):
            print(f"    {k:25s}: {v:.4f}")

    return results


# Main Loop for training QAT Student
all_seeds_results = {}

for seed in SEEDS:
    print(f"  SEED {seed}  [FP32 Teacher → QAT Student KD]")

    train_data, val_data, train_label, val_label = train_test_split(
        data_train, label_train,
        test_size=0.2, shuffle=True, random_state=seed
    )
    training_generator   = DataGenerator(train_data, train_label, n_batch_size)
    validation_generator = DataGenerator(val_data,   val_label,   n_batch_size)

    # Teacher
    teacher_path = os.path.join(SAVE_DIR, f'model_1_seed{seed}.keras')
    teacher_model, teacher_logit_model = load_teacher_with_logits(teacher_path)
    fp32_student = warmup_fp32_student(
        teacher_logit_model = teacher_logit_model,
        training_generator  = training_generator,
        input_shape         = (36, 20, 20, 1),
        n_class             = 6,
        warmup_epochs       = WARMUP_EPOCHS,
        lr                  = LR,
        temperature         = TEMPERATURE_QAT,
        seed                = seed
    )

    # Build QAT student
    qat_student_model = build_qat_student_with_logits(
        input_shape = (36, 20, 20, 1),
        n_class     = 6,
        lr          = LR_QAT)
    print(f"\n  [Weight Transfer] FP32 → QAT...")
    qat_student_model = transfer_fp32_weights_to_qat(
        fp32_model = fp32_student,
        qat_model  = qat_student_model)
    
    dummy   = tf.zeros((2, 36, 20, 20, 1))
    chk_out = qat_student_model(dummy, training=False) 

    if isinstance(chk_out, dict):
        probs_chk = chk_out['probs'].numpy()
    else:
        probs_chk = chk_out.numpy()

    print(f"  [Transfer Check] prob sum: "
          f"{probs_chk.sum(axis=1).mean():.4f} (expect ~1.0)")

    alpha = compute_balanced_alpha_qat(
        teacher_logit_model = teacher_logit_model,
        qat_dual_model      = qat_student_model,  
        training_generator  = training_generator,  
        temperature         = TEMPERATURE_QAT,
        n_batches           = 5
    )

    temperature_tf = tf.constant(TEMPERATURE_QAT, dtype=tf.float32)
    alpha_tf       = tf.constant(alpha,            dtype=tf.float32)

    optimizer = keras.optimizers.RMSprop(learning_rate=LR_QAT)
    train_acc    = tf.keras.metrics.CategoricalAccuracy()
    val_acc      = tf.keras.metrics.CategoricalAccuracy()
    train_loss_m = tf.keras.metrics.Mean()
    val_loss_m   = tf.keras.metrics.Mean()
    qat_train_step = make_qat_train_step(
        teacher_logit_model = teacher_logit_model,
        qat_dual_model      = qat_student_model,   
        optimizer           = optimizer,
        acc_metric          = train_acc,
        loss_metric         = train_loss_m)
    qat_val_step = make_qat_val_step(
        teacher_logit_model = teacher_logit_model,
        qat_dual_model      = qat_student_model,  
        acc_metric          = val_acc,
        loss_metric         = val_loss_m)

    history      = {'train_loss': [], 'train_acc': [],
                    'val_loss'  : [], 'val_acc'  : []}
    best_val_acc = 0.0
    student_npz_path = os.path.join(
        QAT_KD_MODELS_DIR, f'qat_student_seed{seed}.npz')
    save_weights_npz(qat_student_model, student_npz_path)
    print(f"\n  [QAT Training] {QAT_EPOCHS} epochs | "
          f"T={TEMPERATURE_QAT} | α={alpha:.4f} | LR={LR_QAT}")

    for epoch in range(QAT_EPOCHS):
        train_acc.reset_state()
        val_acc.reset_state()
        train_loss_m.reset_state()
        val_loss_m.reset_state()

        for bd, bl in training_generator:
            qat_train_step(bd, bl, temperature_tf, alpha_tf)

        for bd, bl in validation_generator:
            qat_val_step(bd, bl, temperature_tf, alpha_tf)

        t_loss = float(train_loss_m.result())
        v_loss = float(val_loss_m.result())
        t_acc  = float(train_acc.result())
        v_acc  = float(val_acc.result())

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        print(f"  Epoch {epoch+1:02d}/{QAT_EPOCHS} | "
              f"Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | "
              f"Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            save_weights_npz(qat_student_model, student_npz_path)
            print(f"  [Best] val_acc={v_acc:.4f} → {student_npz_path}")
    print(f"\n  [Restoring best] val_acc={best_val_acc:.4f}")
    load_weights_npz(qat_student_model, student_npz_path)

    seed_result = {
        'seed'          : seed,
        'alpha_used'    : float(alpha),
        'temperature'   : TEMPERATURE_QAT,
        'warmup_epochs' : WARMUP_EPOCHS,
        'qat_epochs'    : QAT_EPOCHS,
        'best_val_acc'  : float(best_val_acc),
        'history'       : history,
        'test'  : evaluate_qat_student(qat_student_model,
                                       test_generator,   'test'),
        'test_1': evaluate_qat_student(qat_student_model,
                                       test_1_generator, 'test_1'),
        'test_2': evaluate_qat_student(qat_student_model,
                                       test_2_generator, 'test_2'),
    }
    all_seeds_results[f'seed_{seed}'] = seed_result

for split in ['test', 'test_1', 'test_2']:
    print(f"\n  [{split}]")
    for metric in ['accuracy', 'f1_weighted',
                   'precision_weighted', 'recall_weighted']:
        values = [all_seeds_results[f'seed_{s}'][split][metric]
                  for s in SEEDS]
        print(f"    {metric:25s}: "
              f"mean={np.mean(values):.4f}  "
              f"std={np.std(values):.4f}  "
              f"per_seed={[round(v,4) for v in values]}")

QAT TFLite

In [ ]:
# Export trained QAT Student to tflite
def representative_data_gen(data_train, batch_size=32):
    num_samples = data_train.shape[0]
    shuffled_indices = np.arange(num_samples)
    np.random.shuffle(shuffled_indices)
    
    for i in range(0, num_samples, batch_size):
        batch_indices = shuffled_indices[i:i + batch_size]
        yield [data_train[batch_indices].astype(np.float32)]


# export_qat_tflite
def export_qat_tflite(
        quant_aware_model,
        save_path,
        data_train,        
        keep_float_io=True):

    print(f"\n  [QAT→TFLite] Exporting...")

    converter = tf.lite.TFLiteConverter.from_keras_model(
        quant_aware_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter.representative_dataset = lambda: \
        representative_data_gen(data_train)
    converter.inference_input_type  = tf.float32
    converter.inference_output_type = tf.float32
    converter._experimental_lower_tensor_list_ops = False

    try:
        tflite_model = converter.convert()
        with open(save_path, 'wb') as f:
            f.write(tflite_model)
        size_mb = os.path.getsize(save_path) / (1024**2)
        print(f"  Saved : {save_path}")
        print(f" Size  : {size_mb:.3f} MB")
        return size_mb
    except Exception as e:
        print(f"  [ERROR] TFLite export failed: {e}")
        return -1

# Evakuate QAT tflite Student
def evaluate_tflite(tflite_path, test_data, test_labels,
                    label_name, n_warmup=5):

    if not os.path.exists(tflite_path):
        print(f"      [SKIP] Not found: {os.path.basename(tflite_path)}")
        return None

    interpreter = tf.lite.Interpreter(model_path  = tflite_path,
                                      num_threads = 1)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype    = input_details[0]['dtype']
    output_dtype   = output_details[0]['dtype']
    in_scale,  in_zp  = input_details[0]['quantization']
    out_scale, out_zp = output_details[0]['quantization']

    def preprocess(sample_f32):
        if input_dtype == np.int8 and in_scale != 0:
            return (sample_f32 / in_scale + in_zp).astype(np.int8)
        return sample_f32

    def postprocess(output):
        if output_dtype == np.int8 and out_scale != 0:
            return (output.astype(np.float32) - out_zp) * out_scale
        return output.astype(np.float32)

    for i in range(min(n_warmup, len(test_data))):
        sample = preprocess(test_data[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()

    preds   = []
    t_start = time.perf_counter()

    for i in range(len(test_data)):
        sample = preprocess(test_data[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(
            output_details[0]['index']).copy()
        preds.append(postprocess(output))

    t_end   = time.perf_counter()
    t_total = t_end - t_start
    n       = len(test_data)
    lat_ms  = (t_total / n) * 1000
    tput    = n / t_total
    y_prob = np.concatenate(preds, axis=0)
    y_true = np.argmax(test_labels, axis=1)
    y_pred = np.argmax(y_prob,      axis=1)

    present_classes = np.union1d(np.unique(y_true), np.unique(y_pred))
    n_classes       = len(present_classes)
    target_names    = [f'Gesture_{c+1}' for c in present_classes]

    acc    = float(accuracy_score(y_true, y_pred))
    prec_w = float(precision_score(y_true, y_pred, average='weighted',
                                   zero_division=0, labels=present_classes))
    rec_w  = float(recall_score(y_true, y_pred, average='weighted',
                                zero_division=0, labels=present_classes))
    f1_w   = float(f1_score(y_true, y_pred, average='weighted',
                             zero_division=0, labels=present_classes))
    prec_m = float(precision_score(y_true, y_pred, average='macro',
                                   zero_division=0, labels=present_classes))
    rec_m  = float(recall_score(y_true, y_pred, average='macro',
                                zero_division=0, labels=present_classes))
    f1_m   = float(f1_score(y_true, y_pred, average='macro',
                             zero_division=0, labels=present_classes))
    kappa  = float(cohen_kappa_score(y_true, y_pred))

    size_mb = os.path.getsize(tflite_path) / (1024 ** 2)

    print(f"      [{label_name}] "
          f"Acc={acc:.4f}  F1w={f1_w:.4f}  F1m={f1_m:.4f}  "
          f"κ={kappa:.4f}  Lat={lat_ms:.2f}ms  Size={size_mb:.3f}MB")

    return {
        "accuracy"                        : round(acc,    6),
        "precision_weighted"              : round(prec_w, 6),
        "recall_weighted"                 : round(rec_w,  6),
        "f1_weighted"                     : round(f1_w,   6),
        "precision_macro"                 : round(prec_m, 6),
        "recall_macro"                    : round(rec_m,  6),
        "f1_macro"                        : round(f1_m,   6),
        "kappa"                           : round(kappa,  6),
        "n_classes_detected"              : int(n_classes),
        "confusion_matrix"                : confusion_matrix(
                                                y_true, y_pred,
                                                labels=present_classes
                                            ).tolist(),
        "classification_report"           : classification_report(
                                                y_true, y_pred,
                                                labels=present_classes,
                                                target_names=target_names,
                                                digits=4,
                                                zero_division=0
                                            ),
        "total_inference_time_s"          : round(t_total, 4),
        "avg_inference_time_per_sample_ms": round(lat_ms,  4),
        "throughput_samples_per_sec"      : round(tput,    2),
        "model_size_mb"                   : round(size_mb, 4)
    }


# Evaluating QAT tf.lite Student
all_qat_tflite_results = {}

for seed in SEEDS:
    print(f"  SEED {seed}  [QAT KD TFLite Export + Eval]")
    train_s, _, train_label_s, _ = train_test_split(
        data_train, label_train,
        test_size    = 0.2,
        shuffle      = True,
        random_state = seed)

    qat_student_model = build_qat_student_with_logits(
        input_shape = (36, 20, 20, 1),
        n_class     = 6,
        lr          = LR_QAT)

    # Load trained weights from .npz ─────────────────────
    student_npz_path = os.path.join(
        QAT_KD_MODELS_DIR, f'qat_student_seed{seed}.npz')
    load_weights_npz(qat_student_model, student_npz_path)

    with quantize_scope({
        'ScaledDotProductAttention'                         : ScaledDotProductAttention,
        'MultiHeadAttention'                                : MultiHeadAttention,
        'TransformerMultiHeadAttentionEncoderLayer'         : TransformerMultiHeadAttentionEncoderLayer,
        'TimeDistributedQuantizeConfig'                     : TimeDistributedQuantizeConfig,
        'TransformerMultiHeadAttentionEncoderQuantizeConfig': TransformerMultiHeadAttentionEncoderQuantizeConfig,
        'DenseQuantizeConfig'                               : DenseQuantizeConfig,
        'DenseLinearQuantizeConfig'                         : DenseLinearQuantizeConfig,
        'ActivationQuantizeConfig'                          : ActivationQuantizeConfig,
    }):
        probs_only_model = tf.keras.Model(
            inputs  = qat_student_model.input,
            outputs = qat_student_model.output['probs'],
            name    = 'qat_probs_only'
        )

    tflite_path = os.path.join(QAT_TFLITE_DIR, f'qat_student_seed{seed}_int8.tflite')

    int8_size_mb = export_qat_tflite(   
        quant_aware_model = probs_only_model,
        save_path         = tflite_path,
        data_train        = train_s,      
        keep_float_io     = True)

    metrics = {}
    if int8_size_mb > 0:
        print(f"\n  [Evaluating TFLite INT8]")
        for ts_name, (ts_data, ts_labels) in test_sets_dict.items():
            metrics[ts_name] = evaluate_tflite(   
                tflite_path = tflite_path,
                test_data   = ts_data,
                test_labels = ts_labels,
                label_name  = f"QAT-INT8|{ts_name}",
                n_warmup    = 5)
    else:
        print("  [SKIP] Export failed — no evaluation")

    # ── 6. Save results ────────────────────────────────────────
    seed_result = {
        'seed'        : seed,
        'int8_size_mb': int8_size_mb,
        'metrics'     : metrics
    }
    all_qat_tflite_results[f'seed_{seed}'] = seed_result
    
for split in ['all_rooms', 'room2', 'room3']:
    print(f"\n  [{split}]")
    for metric in ['accuracy', 'f1_weighted',
                   'precision_weighted', 'recall_weighted']:
        values = [
            all_qat_tflite_results[f'seed_{s}']['metrics'][split][metric]
            for s in SEEDS
            if all_qat_tflite_results[f'seed_{s}']['metrics'].get(split)
        ]
        if values:
            print(f"    {metric:25s}: "
                  f"mean={np.mean(values):.4f}  "
                  f"std={np.std(values):.4f}  "
                  f"per_seed={[round(v,4) for v in values]}")